In [5]:
!pip install -q importlib-resources

import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np

In [6]:
import os

# ---------------------------------------------------------
# Step 1: Load the Online Dataset (TFDS)
# ---------------------------------------------------------
print("Loading dataset from TensorFlow Datasets...")
(train_ds, val_ds, test_ds), metadata = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,  # Yields (image, label) pairs
)

num_classes = metadata.features['label'].num_classes
get_label_name = metadata.features['label'].int2str

# ---------------------------------------------------------
# Step 2: Preprocess & Augment Data
# ---------------------------------------------------------
IMG_SIZE = 160
BATCH_SIZE = 32

def format_example(image, label):
    """Resize image and normalize pixel values to [0, 1]."""
    image = tf.cast(image, tf.float32)
    image = (image / 127.5) - 1.0  # Normalize to [-1, 1] for Transfer Learning
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, label

# Apply preprocessing and set up batching + caching for performance
train_batches = (
    train_ds
    .map(format_example, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

val_batches = (
    val_ds
    .map(format_example, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

# Data Augmentation to prevent overfitting
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.1),
])

# Path to save/load the trained model
MODEL_PATH = 'cats_vs_dogs_mobilenetv2.keras'

if os.path.exists(MODEL_PATH):
    # ---------------------------------------------------------
    # Step 3 & 4 (Alternative): Load existing model
    # ---------------------------------------------------------
    print(f"Found existing model at '{MODEL_PATH}'. Loading without retraining...")
    model = tf.keras.models.load_model(MODEL_PATH)
else:
    # ---------------------------------------------------------
    # Step 3: Build Model (Transfer Learning with MobileNetV2)
    # ---------------------------------------------------------
    print("No pre-trained model found. Building and training a new one...")
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False  # Freeze pre-trained weights

    model = tf.keras.Sequential([
        data_augmentation,
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1)  # Single linear output unit for binary classification
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

    # ---------------------------------------------------------
    # Step 4: Train the Model
    # ---------------------------------------------------------
    EPOCHS = 5
    print("\nStarting Model Training...")
    history = model.fit(
        train_batches,
        epochs=EPOCHS,
        validation_data=val_batches
    )

    # Save the model for subsequent runs
    print(f"Saving trained model to '{MODEL_PATH}'...")
    model.save(MODEL_PATH)

# ---------------------------------------------------------
# Step 5: Evaluate on Test Set
# ---------------------------------------------------------
test_batches = (
    test_ds
    .map(format_example, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
)

loss, accuracy = model.evaluate(test_batches)
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

# ---------------------------------------------------------
# Step 6: Inference Function
# ---------------------------------------------------------
def predict_image_url(url):
    """Utility to download an image from a URL and predict Cat or Dog."""
    path = tf.keras.utils.get_file(origin=url)
    img = tf.keras.utils.load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = tf.keras.utils.img_to_array(img)
    img_array = (img_array / 127.5) - 1.0  # Rescale [-1, 1]
    img_batch = tf.expand_dims(img_array, 0)  # Add batch dimension

    predictions = model.predict(img_batch)
    score = tf.nn.sigmoid(predictions[0]).numpy()[0]

    label = "Dog" if score > 0.5 else "Cat"
    confidence = score if label == "Dog" else 1 - score

    plt.imshow(img)
    plt.title(f"Prediction: {label} ({confidence * 100:.1f}%)")
    plt.axis("off")
    plt.show()

# Example usage:
# predict_image_url("https://storage.googleapis.com/download.tensorflow.org/example_images/320px-Felis_catus-cat_on_snow.jpg")

Loading dataset from TensorFlow Datasets...
No pre-trained model found. Building and training a new one...

Starting Model Training...
Epoch 1/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 348s 574ms/step - accuracy: 0.7860 - loss: 0.4277 - val_accuracy: 0.9420 - val_loss: 0.1638
Epoch 2/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 366s 549ms/step - accuracy: 0.9042 - loss: 0.2177 - val_accuracy: 0.9592 - val_loss: 0.1070
Epoch 3/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 310s 530ms/step - accuracy: 0.9224 - loss: 0.1774 - val_accuracy: 0.9673 - val_loss: 0.0884
Epoch 4/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 317s 542ms/step - accuracy: 0.9328 - loss: 0.1569 - val_accuracy: 0.9703 - val_loss: 0.0783
Epoch 5/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 309s 529ms/step - accuracy: 0.9368 - loss: 0.1473 - val_accuracy: 0.9759 - val_loss: 0.0721
Saving trained model to 'cats_vs_dogs_mobilenetv2.keras'...
73/73 ━━━━━━━━━━━━━━━━━━━━ 29s 394ms/step - accuracy: 0.9819 - loss: 0.0626

Test Accuracy: 98.19%
